In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

https://www.youtube.com/watch?v=V_xro1bcAuA

In [4]:
import torch
import matplotlib.pyplot as plt
print(torch.__version__)

2.1.2+cpu


## Messing around with tensors

In [5]:
scalar = torch.tensor(7)
print(scalar)
print(scalar.ndim)
print(scalar.item())

vector = torch.tensor([7, 7])
print(vector)
print(vector.ndim)
print(vector.shape)

MATRIX = torch.tensor([[7, 8],[9, 10]])
print(MATRIX)
print(MATRIX.ndim)
print(MATRIX.shape)
print(MATRIX[0])

#one "frame" of 3x3 matrix
TENSOR = torch.tensor([[[1,2,3],[4,5,6],[2,3,4]]])
print(TENSOR)
print(TENSOR.ndim)
print(TENSOR.shape)
print(TENSOR[0])

#random tensors (starting weight tensors which are adjusted based on the data)
RANDOM_TENSOR = torch.rand(2,3,4)
print(RANDOM_TENSOR)
print(RANDOM_TENSOR.shape, RANDOM_TENSOR.ndim)

#random tensor with similar shape to image
RANDOM_IMAGE_TENSOR = torch.rand(size=(3,224,224))
print(RANDOM_IMAGE_TENSOR.shape, RANDOM_IMAGE_TENSOR.ndim)

#zeros, ones
ZEROS = torch.zeros(1, 3, 3)
ONES = torch.ones(1, 3, 3)

tensor(7)
0
7
tensor([7, 7])
1
torch.Size([2])
tensor([[ 7,  8],
        [ 9, 10]])
2
torch.Size([2, 2])
tensor([7, 8])
tensor([[[1, 2, 3],
         [4, 5, 6],
         [2, 3, 4]]])
3
torch.Size([1, 3, 3])
tensor([[1, 2, 3],
        [4, 5, 6],
        [2, 3, 4]])
tensor([[[0.8335, 0.9874, 0.5130, 0.5357],
         [0.7951, 0.8151, 0.3828, 0.3820],
         [0.0129, 0.1138, 0.2291, 0.9229]],

        [[0.6434, 0.6693, 0.1721, 0.8725],
         [0.6986, 0.6177, 0.0133, 0.3629],
         [0.9518, 0.7123, 0.8212, 0.4107]]])
torch.Size([2, 3, 4]) 3
torch.Size([3, 224, 224]) 3


## Data Types, Further Tensor Manipulation

float32 is default (applies when dtype=None)

Other options:
-float16
-

In [6]:
ONES = torch.ones(3,3,3)
print(ONES.dtype)

print(torch.range(1,10)) #deprecated [a,b]
print(torch.arange(1,10)) #use this [a,b)
print(torch.arange(1,10,2)) #step size is 3rd arg
range_vec = torch.arange(1,10)
zeros_like_range_vec = torch.zeros_like(input=range_vec) 
print(zeros_like_range_vec)


torch.float32
tensor([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.])
tensor([1, 2, 3, 4, 5, 6, 7, 8, 9])
tensor([1, 3, 5, 7, 9])
tensor([0, 0, 0, 0, 0, 0, 0, 0, 0])


/tmp/ipykernel_33/1008219661.py:4: UserWarning: torch.range is deprecated and will be removed in a future release because its behavior is inconsistent with Python's range builtin. Instead, use torch.arange, which produces values in [start, end).
  print(torch.range(1,10)) #deprecated [a,b]


Evidently, data type is a big sticking point, and often device they're on is confused.

In [7]:
float_32_tensor = torch.tensor(
    [3,6,9], 
    dtype=None,#https://pytorch.org/docs/stable/tensors.html
    device=None, 
    requires_grad=False #track gradients?
)

uint8_tensor = torch.tensor(
    [3,6,9],
    dtype=torch.uint8,
    device=None,
    requires_grad=False
)

print(float_32_tensor)
print(uint8_tensor)

tensor([3, 6, 9])
tensor([3, 6, 9], dtype=torch.uint8)


## Tensor Attributes and Manipulation, 2:08:59 in video and on

In [8]:
SOME_TENSOR = torch.rand([1,3,4], dtype=torch.float16)
print(SOME_TENSOR.shape, SOME_TENSOR.size())
print(f"Datatype: {SOME_TENSOR.dtype}")
print(f"Tensor is on device: {SOME_TENSOR.device}")


torch.Size([1, 3, 4]) torch.Size([1, 3, 4])
Datatype: torch.float16
Tensor is on device: cpu


In [9]:
T = torch.tensor([1,2,3])
print(T+1-2)
print(torch.add(T,torch.sub(1,2)))
print(T*11)
print(torch.mul(T,11))
print(T/2)

#element-wise
print(T*T)

#dotprod
print(torch.matmul(T,T))

#left off at 2:27:27

tensor([0, 1, 2])
tensor([0, 1, 2])
tensor([11, 22, 33])
tensor([11, 22, 33])
tensor([0.5000, 1.0000, 1.5000])
tensor([1, 4, 9])
tensor(14)


## Larger Tensor MatMul

Rules for use: 
1. Common error - shape error, incompatible matrix shapes. Inner dimensions must match, meaning (3,2)@(2,3) and (2,3)@(3,2) are good, but (3,2)@(3,2) and (2,3)@(2,3) won't work. Resulting shape is (outer,outer), so for (3,2)@(2,3), result is 3x3.

In [28]:
print(torch.matmul(torch.rand(4,123),torch.rand(123,4)))

#define two 3x2 tensors
A = torch.tensor([[1, 2],[3, 4],[5, 6]])
B = torch.tensor([[7, 10],[8, 11],[9, 12]])
print("A:",A.shape,"B:",B.shape)

#transpose B (rotate about diagonal right?)
Bt = B.T
print(B)
print(Bt)
print("B, transposed:", Bt.shape)

#i know, I'm prob not following the python style guide here
AdotBshort = torch.mm(A, Bt)
AdotB = torch.matmul(A, Bt)
AdotBShape = AdotB.shape
print(AdotBShape)
print(AdotB)



tensor([[30.1186, 34.1450, 30.4961, 31.8032],
        [26.5376, 34.2375, 26.6382, 30.8173],
        [28.3177, 31.5378, 26.6681, 30.5364],
        [27.4704, 33.8731, 26.5830, 30.3032]])
A: torch.Size([3, 2]) B: torch.Size([3, 2])
tensor([[ 7, 10],
        [ 8, 11],
        [ 9, 12]])
tensor([[ 7,  8,  9],
        [10, 11, 12]])
B, transposed: torch.Size([2, 3])
torch.Size([3, 3])
tensor([[ 27,  30,  33],
        [ 61,  68,  75],
        [ 95, 106, 117]])


## Min, Max, Mean, etc.
Seems that torch.arange generates int64, where mean requires float or complex, see ex. below of recasting

In [37]:
A = torch.arange(0,100,10)
maxA = torch.max(A)
maxA2 = A.max() #this syntax applies to min and mean as well...
minA = torch.min(A)
meanA = torch.mean(A.type(torch.float32))
sumA = torch.sum(A)
print("maxA:", maxA, "maxA2", maxA2)
print("minA:", minA)
print("meanA:", meanA)
print("sumA:", sumA)

maxA: tensor(90) maxA2 tensor(90)
minA: tensor(0)
meanA: tensor(45.)
sumA: tensor(450)


## Argmin, Argmax
"Positional min/max" - what is the index of the min/max/mean etc.?


In [52]:
A = torch.arange(1,100,10)

In [59]:
A.argmin()
A[A.argmin()]

tensor(1)

In [58]:
A.argmax()
A[A.argmax()]

tensor(91)

## Reshaping, staking, squeezing, unsqueezing
Reshaping: redefines shape
View: returns a view of the original tensor but persists original tensor in memory 
Stacking: vstack, hstack concat compatible tensors
Squeezing: removes all x1 dimensions from tensor
Unsqueezing: adds a x1 dimension to tensor
Permute: return view of input with dim permuted (swapped) as defined

In [86]:
A = torch.arange(1., 11.)
print(A, A.shape)

#reshape, add a dim (must be compatible number of elements)
AReshaped1 = A.reshape(10,1)
print(AReshaped1, AReshaped1.shape)
AReshaped2 = A.reshape(5,2)
print(AReshaped2, AReshaped2.shape)

#change the view
AView = A.view(2,5)
print(AView, AView.shape)

#changing AView changes A because they share memory when view is used
A[0]=123
print(A, AView)

#stack tensors with stack/vstack/hstack
AVs = torch.stack([A, A, A], dim=0)
print(AVs)
AHs = torch.stack([A, A, A], dim=1)
print(AHs)

#squeeze/unsqueeze (torch.unsqueeze(A, dim=0), A.unsqueeze(dim=0))
Aunsqueeze = A.unsqueeze(dim=0)
print(Aunsqueeze, Aunsqueeze.shape)
Asqueeze = Aunsqueeze.squeeze()
print(Asqueeze, Asqueeze.shape)

#permute - returns a view with permuted dimensions (i.e. rearranged dims)
I = torch.rand(size=(224,224,3))
Ip = torch.permute(I, (1, 2, 0))
print(I.shape, Ip.shape)

#they share memory, yet the shapes are different. the indices follow the permutation to check the same corresponding element
I[112,31,2]=124124
print(I[112,31,2],Ip[31,2,112])

tensor([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.]) torch.Size([10])
tensor([[ 1.],
        [ 2.],
        [ 3.],
        [ 4.],
        [ 5.],
        [ 6.],
        [ 7.],
        [ 8.],
        [ 9.],
        [10.]]) torch.Size([10, 1])
tensor([[ 1.,  2.],
        [ 3.,  4.],
        [ 5.,  6.],
        [ 7.,  8.],
        [ 9., 10.]]) torch.Size([5, 2])
tensor([[ 1.,  2.,  3.,  4.,  5.],
        [ 6.,  7.,  8.,  9., 10.]]) torch.Size([2, 5])
tensor([123.,   2.,   3.,   4.,   5.,   6.,   7.,   8.,   9.,  10.]) tensor([[123.,   2.,   3.,   4.,   5.],
        [  6.,   7.,   8.,   9.,  10.]])
tensor([[123.,   2.,   3.,   4.,   5.,   6.,   7.,   8.,   9.,  10.],
        [123.,   2.,   3.,   4.,   5.,   6.,   7.,   8.,   9.,  10.],
        [123.,   2.,   3.,   4.,   5.,   6.,   7.,   8.,   9.,  10.]])
tensor([[123., 123., 123.],
        [  2.,   2.,   2.],
        [  3.,   3.,   3.],
        [  4.,   4.,   4.],
        [  5.,   5.,   5.],
        [  6.,   6.,   6.],
        [  7.,

# Numpy Integration


In [9]:
import torch
import numpy as np

array = np.arange(1.0, 8.0)
tensor = torch.from_numpy(array)
array, tensor

(array([1., 2., 3., 4., 5., 6., 7.]),
 tensor([1., 2., 3., 4., 5., 6., 7.], dtype=torch.float64))

In [3]:
array.dtype, tensor.dtype

In [11]:
#distributes the addition
array = array + 1
tensor = tensor + 1
array, tensor

(array([3., 4., 5., 6., 7., 8., 9.]),
 tensor([2., 3., 4., 5., 6., 7., 8.], dtype=torch.float64))

# Reproducibility - random seeds etc

In [16]:
import torch
RANDOM_SEED = 1234

torch.manual_seed(RANDOM_SEED)
tA = torch.rand(3,4)

torch.manual_seed(RANDOM_SEED)
tB = torch.rand(3,4)
print(tA == tB)

tensor([[True, True, True, True],
        [True, True, True, True],
        [True, True, True, True]])


# GPU Access
Have to activate GPU runtime to use this section

In [2]:
import torch
torch.cuda.is_available()

True

In [3]:
!nvidia-smi

Fri Jun 21 19:46:47 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03             Driver Version: 535.129.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla P100-PCIE-16GB           Off | 00000000:00:04.0 Off |                    0 |
| N/A   31C    P0              25W / 250W |      2MiB / 16384MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [ ]:
#device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"


In [4]:
torch.cuda.device_count()

1

In [ ]:
#can use argparse to run code on multiple devices without modification
# import argparse
# import torch

# parser = argparse.ArgumentParser(description="PyTorch Example")
# parser.add_argument('--disable-cuda', action='store_true', help='Disable CUDA')
# args = parser.parse_args()
# args.device = None
# if not args.disable_cuda and torch.cuda.is_available():
#     args.device = torch.device('cuda')
# else:
#     args.device = torch.device('cpu')



# Moving data to GPUs
(can't be numpy transformed while on GPU it seems)

In [ ]:
tensor = torch.tensor([1,2,3], device="cpu")
print(tensor, tensor.device)
device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_tensor = tensor.to(device)
print(gpu_tensor, gpu_tensor.device)

In [ ]:
tensor_back_on_cpu = gpu_tensor.cpu().numpy()